In [ ]:
!pip install tensorflow imbalanced-learn seaborn

In [ ]:
from google.colab import drive
import os

# 1. Prompt Google Colab to connect to your Drive
drive.mount('/content/drive')

# 2. Define the path to your shared folder.
# REPLACE 'NAME_OF_YOUR_FOLDER' with the actual folder name.
folder_path = '/content/drive/MyDrive/NAME_OF_YOUR_FOLDER'

# 3. Change the working directory so relative paths work automatically
os.chdir(folder_path)
print("Current Working Directory:", os.getcwd())

# 4. Verify the directories exist
print("Visible items in this directory:", os.listdir())

In [ ]:
%%writefile src/model.py
from tensorflow.keras import layers, models

def residual_block(x, filters=32, kernel_size=5):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.ReLU()(x)
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    x = layers.MaxPool1D(pool_size=5, strides=2, padding="same")(x)
    return x

def build_kachuee_cnn(input_shape=(187, 1), n_classes=5):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 5, padding="same")(inputs)
    x = layers.ReLU()(x)
    for _ in range(5):
        x = residual_block(x)
    x = layers.Flatten()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return models.Model(inputs, outputs)

In [ ]:
from src.model import build_kachuee_cnn

model = build_kachuee_cnn()

model.summary()


In [ ]:
import os

# Check if both files exist in the shared directory
model_exists = os.path.exists("src/model.py")
data_exists = os.path.exists("src/data.py")

print("src/model.py found:", model_exists)
print("src/data.py found:", data_exists)

# Test importability to ensure there are no syntax errors
if model_exists and data_exists:
    try:
        from src.model import build_kachuee_cnn
        print("Successfully imported build_kachuee_cnn from model.py")

        from src.data import load_and_prepare
        print("Successfully imported load_and_prepare from data.py")

        print("\nSYNC COMPLETE: You and Person B are byte-identical. Proceed to Phase 3.")
    except Exception as e:
        print("\nIMPORT FAILED. Check the files for syntax errors:")
        print(e)
else:
    print("\nFILES MISSING: Wait for Person B to finish uploading src/data.py.")

In [ ]:
from src.data import load_and_prepare
from src.model import build_kachuee_cnn

# This function reads the CSVs from your data/ folder, reshapes them for the 1D CNN,
# and returns the exact train/test splits.
X_train, y_train, X_test, y_test = load_and_prepare()

print("Data loaded successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
import numpy as np
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. FIX: Shuffle X_train and y_train together with a fixed seed
indices = np.arange(X_train.shape[0])
np.random.seed(42)  # Keeps it reproducible
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

# 2. Instantiate the model
model = build_kachuee_cnn()

# 3. Compile the model
model.compile(optimizer=Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 4. Train the model (validation_split will now get a healthy mix of classes)
history = model.fit(
    X_train, y_train, validation_split=0.1, epochs=30, batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True),
               ReduceLROnPlateau(patience=3, factor=0.5)]
)

# 5. Save the trained weights
model.save("results/models/baseline.keras")
print("\nModel training complete and saved to results/models/baseline.keras")

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Generate predictions on the test set
y_pred = model.predict(X_test).argmax(axis=1)

# 2. Generate and save the classification report
report = classification_report(
    y_test,
    y_pred,
    target_names=["N", "S", "V", "F", "Q"],
    output_dict=True
)

acc = report["accuracy"]

with open("results/metrics_baseline.json", "w") as f:
    json.dump(report, f, indent=2)

# 3. Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["N","S","V","F","Q"],
    yticklabels=["N","S","V","F","Q"]
)

plt.title(f"Baseline — Test Accuracy {acc:.3f}")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.savefig(
    "results/figures/confusion_baseline.png",
    dpi=150
)

plt.show()

print("\nEvaluation complete.")
print(f"Overall Test Accuracy: {acc:.4f}")
print("metrics_baseline.json saved.")
print("confusion_baseline.png saved.")

In [ ]:
import json
import pandas as pd

# 1. Load both metric files
with open("results/metrics_baseline.json") as f:
    base = json.load(f)
with open("results/metrics_smote.json") as f:
    smote = json.load(f)

# 2. Extract per-class performance metrics
classes = ["N", "S", "V", "F", "Q"]
rows = []
for cls in classes:
    rows.append({
        "class": cls,
        "precision_baseline": base[cls]["precision"],
        "precision_smote": smote[cls]["precision"],
        "recall_baseline":    base[cls]["recall"],
        "recall_smote":    smote[cls]["recall"],
        "f1_baseline":        base[cls]["f1-score"],
        "f1_smote":        smote[cls]["f1-score"],
        "support":            base[cls]["support"],
    })

# 3. Create DataFrame and calculate differences
df = pd.DataFrame(rows)
df["f1_delta"] = df["f1_smote"] - df["f1_baseline"]
df["recall_delta"] = df["recall_smote"] - df["recall_baseline"]

# 4. Save to the results folder as your source of truth
df.to_csv("results/comparison_table.csv", index=False)

print("--- Overall Accuracy ---")
print(f"Baseline Accuracy: {base['accuracy']:.4f}")
print(f"SMOTE Accuracy:    {smote['accuracy']:.4f}\n")

print("--- Per-Class Breakdown Table ---")
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(classes))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))

# Plot baseline vs SMOTE side by side
ax.bar(x - width/2, df["f1_baseline"], width, label="Baseline", color="steelblue")
ax.bar(x + width/2, df["f1_smote"],    width, label="Baseline + SMOTE", color="seagreen")

# Formatting elements
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("F1-score")
ax.set_title("Per-class F1 Chart: Baseline vs. SMOTE")
ax.set_ylim(0, 1.1)
ax.legend(loc="upper right")

# Annotate bars with their exact values
for i, (b, s) in enumerate(zip(df["f1_baseline"], df["f1_smote"])):
    ax.text(i - width/2, b + 0.01, f"{b:.2f}", ha="center", va="bottom", fontsize=9)
    ax.text(i + width/2, s + 0.01, f"{s:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("results/figures/f1_comparison.png", dpi=150)
plt.show()
print("Grouped F1 bar chart saved to results/figures/f1_comparison.png")

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Loading models and evaluating test set... (this may take a moment)")

# 1. Load both saved models from the shared folder
model_base = load_model("results/models/baseline.keras")
model_smote = load_model("results/models/smote.keras")

# 2. Re-generate predictions to ensure matrix accuracy
y_pred_base = model_base.predict(X_test).argmax(axis=1)
y_pred_smote = model_smote.predict(X_test).argmax(axis=1)

# 3. Compute matrices
cm_base = confusion_matrix(y_test, y_pred_base)
cm_smote = confusion_matrix(y_test, y_pred_smote)

# 4. Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues", ax=axes[0], xticklabels=classes, yticklabels=classes)
axes[0].set_title(f"Baseline Confusion Matrix (Acc: {base['accuracy']:.4f})")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cm_smote, annot=True, fmt="d", cmap="Greens", ax=axes[1], xticklabels=classes, yticklabels=classes)
axes[1].set_title(f"SMOTE Confusion Matrix (Acc: {smote['accuracy']:.4f})")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig("results/figures/confusion_side_by_side.png", dpi=150)
plt.show()
print("Side-by-side confusion matrix image saved to results/figures/confusion_side_by_side.png")